In [ ]:
!pip install -q transformers gTTS pillow deep-translator

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.3/42.3 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.2/98.2 kB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.8/56.8 kB 4.4 MB/s eta 0:00:00


In [ ]:
# Ek kütüphaneler (widget ve gradio)
!pip install -q ipywidgets gradio

import ipywidgets as widgets
from IPython.display import display, clear_output, Audio
import datetime
import os

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 15.4 MB/s eta 0:00:00


In [ ]:
# Modeli yükleleme (BLIP-large)
from transformers import BlipProcessor, BlipForConditionalGeneration
from PIL import Image
from gtts import gTTS
from deep_translator import GoogleTranslator
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"

print("Model (BLIP-large) yükleniyor.")

processor = BlipProcessor.from_pretrained("Salesforce/blip-image-captioning-large")
model = BlipForConditionalGeneration.from_pretrained(
    "Salesforce/blip-image-captioning-large",
    torch_dtype=torch.float16 if device == "cuda" else torch.float32
).to(device)

print("BLIP-large modeli yüklendi.")

⌛ Daha güçlü model (large) yükleniyor... Bu biraz daha uzun sürebilir.


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


preprocessor_config.json:   0%|          | 0.00/445 [00:00<?, ?B/s]

The image processor of type `BlipImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


config.json: 0.00B [00:00, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.88G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/616 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie text_decoder.bert.embeddings.word_embeddings.weight to text_decoder.cls.predictions.decoder.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
BlipForConditionalGeneration LOAD REPORT from: Salesforce/blip-image-captioning-large
Key                                       | Status     |  | 
------------------------------------------+------------+--+-
text_decoder.bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ BLIP-large modeli yüklendi. Artık daha doğru ve detaylı caption üretecek.


In [ ]:
# Ses ve metin ayarları (Türkçe'ye çevirmek, uzunluğunu ayarlamak)
def process_image_final(image_path, detailed=True):
    if not os.path.exists(image_path):
        print(f"Dosya bulunamadı: {image_path}")
        return

    raw_image = Image.open(image_path).convert('RGB')
    display(raw_image)
    print("İşlemler başlatılıyor...")

    inputs = processor(raw_image, return_tensors="pt").to(device)

    # Daha doğru caption için gelişmiş parametreler
    gen_kwargs = {
        "max_length": 80,
        "num_beams": 8 if detailed else 5,
        "repetition_penalty": 1.4,
        "length_penalty": 1.1,
        "early_stopping": True,
        "no_repeat_ngram_size": 3,
        "num_return_sequences": 1
    }

    out = model.generate(**inputs, **gen_kwargs)
    caption_en = processor.decode(out[0], skip_special_tokens=True).strip()

    print(f"🇬🇧 İngilizce: {caption_en}")

    # Çeviri iyileştirmesi: Daha doğal Türkçe için
    try:
        translator = GoogleTranslator(source='en', target='tr')
        caption_tr = translator.translate(caption_en)

        # Küçük Türkçe düzeltmeler (opsiyonel)
        caption_tr = caption_tr.replace("bir bir", "bir").replace("  ", " ")
        print(f"🇹🇷 Türkçe: {caption_tr}")
    except Exception as e:
        print(f"Çeviri hatası: {e}")
        caption_tr = caption_en

    # Kaydetme
    timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
    base_name = os.path.splitext(os.path.basename(image_path))[0]

    audio_filename = f"{base_name}_{timestamp}.mp3"
    txt_filename = f"{base_name}_{timestamp}.txt"

    try:
        tts = gTTS(text=caption_tr, lang='tr', slow=False)
        tts.save(audio_filename)

        with open(txt_filename, "w", encoding="utf-8") as f:
            f.write(f"İngilizce: {caption_en}\n\nTürkçe: {caption_tr}\n\nTarih: {datetime.datetime.now()}")

        display(Audio(audio_filename, autoplay=True))
        print(f"Ses ve metin kaydedildi!")
        print(f"   {txt_filename}")
        print(f"   {audio_filename}")
    except Exception as e:
        print(f"Ses hatası: {e}")

In [ ]:
# Resmi yükleyip metin ses dosyası üretme
uploader = widgets.FileUpload(
    accept='image/*',
    multiple=False,
    description='Resim Yükle (Tıkla veya sürükle-bırak)'
)

output_area = widgets.Output()

def on_upload_change(change):
    if uploader.value:
        uploaded_file = list(uploader.value.values())[0]
        file_name = uploaded_file['metadata']['name']

        save_path = file_name
        with open(save_path, 'wb') as f:
            f.write(uploaded_file['content'])

        with output_area:
            clear_output()
            print(f"{file_name} yüklendi. İşleniyor...")
            process_image_final(save_path, detailed=False)

uploader.observe(on_upload_change, names='value')

display(uploader)
display(output_area)

FileUpload(value={}, accept='image/*', description='📸 Resim Yükle (Tıkla veya sürükle-bırak)')

Output()

In [ ]:
# Aynı işlemi gradio üzerinden yapma (Resmi yükleyip metin ses dosyası üretme)
import gradio as gr

def gradio_process(image, detailed=False):
    if image is None:
        return "Lütfen bir resim yükleyin."

    temp_path = "temp_gradio_image.jpg"
    image.save(temp_path)
    process_image_final(temp_path, detailed=detailed)
    return "İşlem tamamlandı! Ses ve metin dosyaları kaydedildi."

gr.Interface(
    fn=gradio_process,
    inputs=[
        gr.Image(type="pil", label="Resmini Yükle"),
        gr.Checkbox(label="Daha Detaylı Açıklama", value=False)
    ],
    outputs=gr.Textbox(label="Durum Bilgisi"),
    title="AI Görüntü Açıklayıcı + Türkçe Ses",
    description="Resim yükleyin → BLIP ile açıklama yazsın → Türkçe seslendirsin",
    allow_flagging="never"
).launch(share=True)

/usr/local/lib/python3.12/dist-packages/gradio/interface.py:415: UserWarning: The `allow_flagging` parameter in `Interface` is deprecated. Use `flagging_mode` instead.
  warnings.warn(


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://92a9884ee22a57a0fc.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
